# CHIT No. 22 — Mall Customers Clustering

Clusters customers using Annual Income and Spending Score. It loads the Mall Customers CSV if you upload it, otherwise uses a small illustrative sample so the notebook runs immediately. Cluster names are assigned by comparing each cluster's mean income and spending score.

**How to use:** Run each cell from top to bottom. Read the short explanation before every code cell.

### What this cell does
Import the required libraries.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

### What this cell does
Try to load the common Mall Customers CSV file. If it is not present, create a small sample dataset so you can practise immediately.

In [ ]:
try:
    customers = pd.read_csv("Mall_Customers.csv")
    print("Loaded Mall_Customers.csv")
except FileNotFoundError:
    print("Mall_Customers.csv not found; using an illustrative sample dataset.")
    customers = pd.DataFrame({
        "Annual Income (k$)": [15, 18, 20, 22, 25, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100, 110],
        "Spending Score (1-100)": [39, 6, 15, 35, 80, 75, 40, 42, 50, 55, 45, 60, 20, 25, 75, 85, 15, 10, 90, 95]
    })

customers.head()

### What this cell does
Select the two requested features. The common dataset uses these exact column names.

In [ ]:
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X = customers[features].dropna().copy()
X.head()

### What this cell does
Scale both features so income and spending score have a comparable influence on distance.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### What this cell does
Fit K-means with K = 5, a common choice for this two-feature Mall Customers example, and store the cluster IDs.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
X["Cluster"] = kmeans.fit_predict(X_scaled)
X.head()

### What this cell does
Calculate the mean annual income and spending score for each cluster.

In [ ]:
cluster_means = X.groupby("Cluster")[features].mean()
print("Mean values for each cluster:")
print(cluster_means.round(2))

### What this cell does
Give each cluster a readable name based on whether its average income and spending score are high or low relative to the overall averages.

In [ ]:
income_mid = X[features[0]].mean()
spending_mid = X[features[1]].mean()

def name_cluster(row):
    income = row[features[0]]
    spending = row[features[1]]
    income_label = "High Income" if income >= income_mid else "Low Income"
    spending_label = "High Spending" if spending >= spending_mid else "Low Spending"
    return income_label + " - " + spending_label

cluster_names = {cluster_id: name_cluster(row)
                 for cluster_id, row in cluster_means.iterrows()}

X["Cluster Name"] = X["Cluster"].map(cluster_names)
print("Cluster names:")
for cluster_id, name in cluster_names.items():
    print(f"Cluster {cluster_id}: {name}")

print("\nCustomers with cluster labels:")
X.head(10)

### What this cell does
Plot customers by annual income and spending score, coloured by cluster, and label each cluster's mean point.

In [ ]:
plt.figure(figsize=(9, 6))
for cluster_id, group in X.groupby("Cluster"):
    plt.scatter(group[features[0]], group[features[1]], s=55, label=cluster_names[cluster_id])

for cluster_id, row in cluster_means.iterrows():
    plt.scatter(row[features[0]], row[features[1]], marker="X", s=220,
                c="black", edgecolor="white")
    plt.annotate(f"C{cluster_id}", (row[features[0]], row[features[1]]),
                 xytext=(5, 5), textcoords="offset points")

plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.title("Mall Customers — K-means Clustering")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()